In [48]:
import torch
import torch.nn as nn
import random

# ---- 数据准备 ----
pairs = [("123", "一二三"), ("456", "四五六"), ("789", "七八九"),
         ("12", "一二"), ("345", "三四五"), ("678", "六七八"),
         ("91", "九一"), ("234", "二三四"), ("567", "五六七"),
         ("890", "八九零"), ("111", "一一一"), ("222", "二二二")]

# 特殊token
PAD = 0
SOS = 1
EOS = 2

src_chars = sorted(set(c for s, _ in pairs for c in s))
tgt_chars = sorted(set(c for _, t in pairs for c in t))

src2idx = {c: i + 1 for i, c in enumerate(src_chars)}  # 0=padding
# tgt：0=PAD,1=SOS,2=EOS，字符从3开始
tgt2idx = {c: i + 3 for i, c in enumerate(tgt_chars)}
idx2tgt = {v: k for k, v in tgt2idx.items()}

src_vocab_size = len(src2idx) + 1
tgt_vocab_size = len(tgt2idx) + 3  # PAD+SOS+EOS

# ---- 模型定义 ----
class Seq2Seq(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, embed_dim=32, hidden=64):
        super().__init__()
        self.src_embed = nn.Embedding(src_vocab, embed_dim)
        self.tgt_embed = nn.Embedding(tgt_vocab, embed_dim)
        self.encoder = nn.GRU(embed_dim, hidden, batch_first=True)
        self.decoder = nn.GRU(embed_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, tgt_vocab)

    def forward(self, src, tgt):
        # Encoder
        _, hidden = self.encoder(self.src_embed(src))
        # Decoder
        output, _ = self.decoder(self.tgt_embed(tgt), hidden)
        return self.fc(output)  # (batch, tgt_len, tgt_vocab)

    def translate(self, src, max_len=10):
        """推理：自回归逐步生成，第一步喂SOS"""
        _, hidden = self.encoder(self.src_embed(src))
        # 【关键】起始输入是SOS，不再是PAD 0
        input_tok = torch.full((src.size(0),1), fill_value=SOS, dtype=torch.long, device=src.device)
        result = []
        for _ in range(max_len):
            output, hidden = self.decoder(self.tgt_embed(input_tok), hidden)
            pred = self.fc(output[:, -1, :]).argmax(dim=-1, keepdim=True)
            result.append(pred)
            input_tok = pred
            if (pred == EOS).all():  # 遇到 EOS 停止
                break
        return torch.cat(result, dim=1)

# ---- 训练 ----
def encode_pair(src, tgt):
    src_ids = [src2idx[c] for c in src]
    # target序列：SOS + 真实字符 + EOS
    tgt_ids = [SOS] + [tgt2idx[c] for c in tgt] + [EOS]
    return torch.tensor([src_ids]), torch.tensor([tgt_ids])

model = Seq2Seq(src_vocab_size, tgt_vocab_size)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(200):
    total_loss = 0
    random.shuffle(pairs)
    for src_text, tgt_text in pairs:
        src_ids, tgt_ids = encode_pair(src_text, tgt_text)
        # tgt输入：去掉末尾EOS；tgt标签：去掉开头SOS
        output = model(src_ids, tgt_ids[:, :-1])
        loss = criterion(output.squeeze(0), tgt_ids[:, 1:].squeeze(0))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1}, Loss: {total_loss / len(pairs):.4f}")

# ---- 测试 ----
model.eval()
with torch.no_grad():
    for src_text, expected in pairs[:4]:
        src_ids = torch.tensor([[src2idx[c] for c in src_text]])
        result = model.translate(src_ids)
        decoded = "".join(
            idx2tgt.get(i.item(), "?")
            for i in result[0]
            if i.item() not in (PAD, SOS, EOS)
        )
        print(f"{src_text} → {decoded} (期望: {expected})")


Epoch 50, Loss: 0.0303
Epoch 100, Loss: 0.0070
Epoch 150, Loss: 0.0031
Epoch 200, Loss: 0.0017
890 → 八九零 (期望: 八九零)
222 → 二二二 (期望: 二二二)
91 → 九一 (期望: 九一)
789 → 七八九 (期望: 七八九)


In [49]:
print(model)

Seq2Seq(
  (src_embed): Embedding(11, 32)
  (tgt_embed): Embedding(13, 32)
  (encoder): GRU(32, 64, batch_first=True)
  (decoder): GRU(32, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=13, bias=True)
)
